# TP : Partie Cyber + Matching Elasticsearch

Ce notebook regroupe **deux parties indépendantes** :

1. **Partie Cyber** — Tests de connectivité réseau (FQDN, DNS, TCP, HTTPS) sur des sites publics (badssl.com, space-match.net).
2. **Partie Matching** — Découverte du matching par vecteurs avec Elasticsearch (index, bulk, script_score) et bonus sur la pondération.


## 0. Préparation (commune)

- Ce dépôt cloné en local dans `TP_matching`.
- Un interpréteur Python pour la partie Matching ; pour la **partie Cyber**, les cellules sont en **PowerShell** (choisir le noyau PowerShell pour exécuter les cellules de la section 1, ou les copier dans un terminal PowerShell).

**Important :** pour la partie Matching, tu dois créer un fichier `.env` à la racine de `TP_matching` avec `ES_URL` et `API_KEY` qui pointent vers ton cluster Elasticsearch (par exemple un déploiement Elastic Cloud). Le notebook se connecte uniquement via ces variables.

**Si tu es dans un Codespace / environnement Linux sans PowerShell**, tu peux :
- soit lire la partie Cyber et l’exécuter plus tard sur un poste Windows avec PowerShell ;
- soit utiliser les **équivalents Linux** donnés juste en dessous (section "Notes pour Linux / Codespace").

In [ ]:
import sys
print("Python:", sys.version)

## 1. Partie Cyber : tests de connectivité 

### 1.1. Le FQDN (Fully Qualified Domain Name)

Pour joindre un serveur sur Internet, on utilise en général un **nom de domaine complet** (FQDN) (ex **badssl.com**) plutôt que son adresse IP. On va voir comment tester la résolution DNS, la connexion TCP et HTTPS.

### 1.2. Adresse IP et résolution DNS

Les machines sur le réseau sont identifiées par une **adresse IP**. Le **DNS** (Domain Name System) fait le lien entre un nom de domaine (FQDN) et une ou plusieurs adresses IP : c’est la **résolution DNS**. La commande **nslookup** permet d’obtenir l’IP associée à un nom.

Copiez/collez la commande ci-dessous dans un terminal **PowerShell** (elle n’est pas exécutée dans le notebook).

```powershell
  nslookup badssl.com
```
Ou sur linux : 

  ```bash
  sudo apt-get update
  sudo apt-get install dnsutils -y
  ```

> **Question :** à ton avis, que se passe‑t‑il si tu lances `nslookup` sur un nom inexistant (par exemple `does-not-exist.space-match.net`) ? Que peux‑tu en déduire sur le rôle du DNS dans la chaîne de connexion ?

### 1.3. Connexion TCP et port 443

Une fois l’IP connue, il faut établir une **connexion TCP** vers le bon **port**. Pour le trafic HTTPS, le port standard est **443**. **Test-NetConnection** permet de vérifier si la machine peut ouvrir une connexion TCP vers ce port.

En **sécurité réseau**, une politique courante est de **bloquer** certains ports. Si on lance **Test-NetConnection** sans préciser le port, c’est le **ping (ICMP)** qui est testé ; un pare-feu peut autoriser le port 443 mais bloquer le ping.

```powershell
Test-NetConnection badssl.com -Port 443
```

Sans port = test ICMP (ping).

```powershell
Test-NetConnection badssl.com
```

Sur Linux: 

  ```bash
  timeout 1 bash -c '</dev/tcp/badssl.com/443 && echo PORT OPEN || echo PORT CLOSED'
  ```

> **Question :** si `Test-NetConnection badssl.com -Port 443` réussit mais que `Test-NetConnection badssl.com` (sans port) échoue, qu’est‑ce que cela t’indique sur les flux autorisés / bloqués par le pare‑feu entre ta machine et le serveur ?

### 1.4. HTTPS et Invoke-WebRequest

Le protocole **HTTPS** assure le chiffrement et l’authentification du serveur (certificat). Pour tester qu’une URL HTTPS répond correctement en PowerShell : 

```powershell
Invoke-WebRequest -Uri "https://badssl.com" -UseBasicParsing
```

Sur Linux : 
  ```bash
  curl -v https://badssl.com
  ```

> **Question :** en observant les entêtes HTTP et les informations TLS retournées par `Invoke-WebRequest` ou `curl -v`, quels éléments te permettent de vérifier que tu parles bien au bon serveur et que la connexion est chiffrée (nom dans le certificat, protocole, code HTTP, etc.) ?

### 1.5. Exercices guidés : space-match.net

On reprend exactement la même démarche que pour `badssl.com`, mais cette fois sur le site `space-match.net`.

1. **Résolution DNS**  
   - Compare les IP de `space-match.net` et `staging.space-match.net` 
   - Question : **les deux noms pointent‑ils vers la même IP ou vers des IP différentes ? Qu’est‑ce que cela te suggère sur la façon dont les environnements (prod / staging) sont séparés ?**

2. **Connexion TCP / port 443**  
   - Teste la connexion TCP vers `space-match.net` sur le port **443**.  
   - Refais le test **sans préciser le port** (ping/ICMP).  
   - Question : **que déduis‑tu de l’état du pare‑feu ?**

3. **HTTPS et certificat**  
   - Lance une requête HTTPS (PowerShell : `Invoke-WebRequest`, Linux : `curl -v https://space-match.net`).  
   - Observe le CN / SAN du certificat et le code HTTP de réponse.  
   - Questions :  
     - **le certificat présenté correspond‑il bien au nom `space-match.net` ?**  
     - **que verrais‑tu dans le navigateur si le certificat ne correspondait pas au FQDN (erreur TLS typique) ?**

## 2. Partie Matching : Elasticsearch en local et index de 4 produits

On travaille avec un **Elasticsearch lancé en local** (sans Docker ou avec Docker), sans ressource distante. L’index `products` contient **4 produits** avec :
- un champ vecteur principal `product_vector` (dims 4),
- un champ `weights_vector` (dims 4).

**Prérequis :** avoir exécuté une fois `install/install.ps1` (Windows) ou `install/install.sh` (Linux/macOS) : Elasticsearch est alors installé en local, démarré en arrière-plan et l’index `products` (4 produits) est déjà créé.
L'index et les 4 produits sont déjà créés par le script d'installation ; inutile de refaire un PUT ou un bulk.

Le **mapping** de l'index définit les champs suivants : 
- **`name`** (texte, type `text`) 
- **`category`** (catégorie fixe, type `keyword`)
- **`product_vector`** (vecteur dense de dimension 4, utilisé pour le matching)
-  **`weights_vector`** (vecteur dense de dimension 4, pour la pondération). 
```json
{
  "mappings": {
    "properties": {
      "name": {
        "type": "text"
      },
      "category": {
        "type": "keyword"
      },
      "product_vector": {
        "type": "dense_vector",
        "dims": 4
      },
      "weights_vector": {
        "type": "dense_vector",
        "dims": 4
      }
    }
  }
}


### 2.1 Elastic Search Client 

### 2.1 Elastic Search Mapping

In [ ]:
### 2.1 Elastic Search Client 

### 2.1. Requête KNN simple
Elasticsearch propose une recherche de type **k‑plus proches voisins (KNN)** sur des champs de type `dense_vector`. L’idée est de représenter chaque document (ici, un produit) par un **vecteur numérique** dans un espace de dimension fixe, puis de chercher, pour un **vecteur de requête**, les *k* documents les plus proches dans cet espace.
Dans notre cas, chaque document de l’index `products` contient un champ vecteur `product_vector` de dimension 4 ; il encode de façon très simplifiée la position du produit dans un « espace de similarité ». Le script ci‑dessous illustre une requête KNN minimale :
- **Connexion au cluster**  
  - `ES_URL = "http://localhost:9200"` : URL de la couche HTTP d’Elasticsearch (un seul nœud, lancé localement par le script d’installation).
- **Vecteur de requête**  
  - `query_vector = [1.0, 0.0, 0.0, 0.0]` : vecteur qui représente le « profil » que l’on cherche (par exemple, un type de produit de référence).  
- **Corps de la requête KNN (`knn_body`)**  
  - `"field": "product_vector"` : champ `dense_vector` sur lequel on fait la recherche de similarité.  
  - `"query_vector": query_vector` : vecteur de requête envoyé à Elasticsearch.  
  - `"k": 4` : nombre de **voisins** que l’on souhaite récupérer (ici les 4 produits les plus proches).  
  - `"num_candidates": 10` : nombre maximum de **candidats** à considérer dans l’index avant de ne garder que les `k` meilleurs. C’est un compromis entre **performance** (réponse rapide) et **qualité** (proximité exacte).
- **Envoi de la requête HTTP**  
  - On construit une requête `POST` vers `"{ES_URL}/products/_search"` avec un corps JSON (`json.dumps(knn_body).encode("utf-8")`) et un header `Content-Type: application/json`.  
  - On utilise `urllib.request.urlopen(req)` pour envoyer la requête et lire la réponse JSON d’Elasticsearch.
- **Lecture des résultats**  
  - La réponse contient un objet `hits` ; on parcourt `result["hits"]["hits"]` et, pour chaque document, on affiche au minimum :
    - `_source["name"]` : le nom du produit ;
    - `_score` : le score de similarité (plus le score est élevé, plus le produit est proche du vecteur de requête selon la métrique choisie).
Sous le capot, Elasticsearch utilise des index de vecteurs (par exemple basés sur des graphes **HNSW**) pour approximativement retrouver les plus proches voisins (**ANN, Approximate Nearest Neighbors**). La **métrique** (distance euclidienne, similarité cosinus, etc.) et la méthode exacte sont configurées dans le mapping / les paramètres de l’index. Pour plus de détails :
- Docs `dense_vector` (champ vecteur) :  
  <https://www.elastic.co/guide/en/elasticsearch/reference/current/dense-vector.html>  
- Docs recherche KNN (`knn` dans `_search`) :  
  <https://www.elastic.co/guide/en/elasticsearch/reference/current/knn-search.html>  
- Vue d’ensemble de la recherche vectorielle / sémantique :  
  <https://www.elastic.co/guide/en/elasticsearch/reference/current/semantic-search.html>


In [7]:
import os
import json
import urllib.request
from pathlib import Path

from dotenv import load_dotenv  # <- python-dotenv


def load_es_config() -> tuple[str, str]:
    env_path = Path(".env")
    load_dotenv(dotenv_path=env_path)
    es_url = os.getenv("ES_URL")
    api_key = os.getenv("API_KEY")
    if not es_url or not api_key:
        raise RuntimeError("ES_URL ou API_KEY manquant dans .env")
    return es_url.rstrip("/"), api_key


ES_URL, ES_API_KEY = load_es_config()


def es_request(method: str, endpoint: str, body: dict | None = None) -> dict:
    if not endpoint.startswith("/"):
        endpoint = "/" + endpoint
    url = ES_URL + endpoint
    data = json.dumps(body).encode("utf-8") if body is not None else None
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"ApiKey {ES_API_KEY}",
    }
    req = urllib.request.Request(url, data=data, headers=headers, method=method.upper())
    with urllib.request.urlopen(req) as resp:
        return json.load(resp)

In [3]:
# Vérifier rapidement les documents indexés dans temp_tp_matching
result = es_request("GET", "/temp_tp_matching/_search?size=4")
for hit in result.get("hits", {}).get("hits", []):
    print(hit["_source"]["name"], "— score:", hit.get("_score"))

HTTPError: HTTP Error 503: Service Unavailable

### 2.2 Requête KNN avec filtre



In [ ]:
# Requête KNN avec filtre sur la catégorie (cluster .env, index temp_tp_matching)
query_vector = [1.0, 0.0, 0.0, 0.0]
knn_with_filter_body = {
    "knn": {
        "field": "product_vector",
        "query_vector": query_vector,
        "k": 4,
        "num_candidates": 10,
        "filter": {
            "term": {"category": "shoes"}
        },
    }
}
result = es_request("POST", "/temp_tp_matching/_search", knn_with_filter_body)
for hit in result.get("hits", {}).get("hits", []):
    print(hit["_source"]["name"], "— score:", hit.get("_score"))

## 3. Requête de matching avec `script_score`

On utilise un script Painless pour calculer un score à partir :
- d’un vecteur d’entrée `params.input_vector`,
- du vecteur stocké dans `doc['product_vector']`,
- (optionnel) d’un vecteur de poids `params.weights`.

Une idée simple consiste à :
1. calculer une distance (par ex. euclidienne),
2. transformer cette distance en score, par exemple :  
   \( score = \frac{1}{1 + distance} \).

In [ ]:
query_body = {
    "query": {
        "script_score": {
            "query": {"match_all": {}},
            "script": {
                "source": """
double distance = 0.0;
for (int i = 0; i < params.input_vector.length; ++i) {
  double diff = params.input_vector[i] - doc['product_vector'][i];
  distance += diff * diff;
}
distance = Math.sqrt(distance);
double score = 1.0 / (1.0 + distance);
return score;
                """,
                "params": {
                    "input_vector": [1.0, 0.0, 0.0, 0.0]
                }
            }
        }
    }
}

print(json.dumps(query_body, indent=2, ensure_ascii=False))

## 4. Bonus : pondération construite et statistiques online

Idées de questions :
1. Comment choisir une pondération par dimension (ex. en fonction de la variance) ?
2. Comment standardiser les features \((x - \mu)/\sigma\) ?
3. Comment mettre à jour \(\mu\) et \(\sigma\) au fil de l’eau (algorithme online) ?

Pseudo-code (Welford) pour une seule dimension, à généraliser à tout le vecteur :

```text
count = count + 1
delta = value - mean
mean = mean + delta / count
delta2 = value - mean
M2 = M2 + delta * delta2

if count > 1:
    variance = M2 / (count - 1)
    std = sqrt(variance)
```

Proposez ensuite une stratégie de pondération des dimensions à partir de ces statistiques (par exemple donner plus de poids aux dimensions les plus discriminantes).